In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

config.py

In [2]:
%%writefile config.py
import os
import torch


def _int(name, default):
    return int(os.environ.get(name, default))


def _flt(name, default):
    return float(os.environ.get(name, default))


SEED = _int("NS_SEED", 42)

DATA_DIR = os.environ.get("NS_DATA_DIR", "data")
TRAIN_PAIRS_PATH = os.path.join(DATA_DIR, "train_pairs.json")
VAL_EVAL_PATH    = os.path.join(DATA_DIR, "val_eval.json")
TEST_EVAL_PATH   = os.path.join(DATA_DIR, "test_eval.json")
PASSAGES_PATH     = os.path.join(DATA_DIR, "passages.json") 
QUERIES_META_PATH = os.path.join(DATA_DIR, "queries_meta.json")
TOKENIZER_PATH = os.environ.get("NS_TOKENIZER", "tokenizer.json")
MODEL_PATH = os.environ.get("NS_MODEL", "best_model.pt")

# Run 2 data: 50k -> 200k MS MARCO examples (~4x more training triples).
# This is the highest-leverage change for retrieval quality.
MAX_EXAMPLES = _int("NS_MAX_EXAMPLES", 200_000)
VOCAB_SIZE = _int("NS_VOCAB_SIZE", 8000)

D_MODEL = _int("NS_D_MODEL", 256)
NHEAD = _int("NS_NHEAD", 4)
NUM_LAYERS = _int("NS_NUM_LAYERS", 4)
DIM_FF = _int("NS_DIM_FF", 512)
DROPOUT = _flt("NS_DROPOUT", 0.1)
MAX_LEN = _int("NS_MAX_LEN", 128)
MODEL_MAX_LEN = _int("NS_MODEL_MAX_LEN", 512)

PRETRAIN_EPOCHS = _int("NS_PRETRAIN_EPOCHS", 3)
PRETRAIN_BS = _int("NS_PRETRAIN_BS", 64)
PRETRAIN_LR = _flt("NS_PRETRAIN_LR", 1e-4)
FINETUNE_EPOCHS = _int("NS_FINETUNE_EPOCHS", 5)
FINETUNE_BS = _int("NS_FINETUNE_BS", 32)
FINETUNE_LR = _flt("NS_FINETUNE_LR", 1e-4)
TEMPERATURE = _flt("NS_TEMPERATURE", 0.05)

EVAL_K = _int("NS_EVAL_K", 10)
ENCODE_BS   = _int("NS_ENCODE_BS",   512)

DEVICE = os.environ.get("NS_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

Writing config.py


model.py

In [3]:
%%writefile model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def masked_mean(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.nhead = nhead
        self.d_head = d_model // nhead
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask):
        B, L, d = x.shape
        def split(t):
            return t.view(B, L, self.nhead, self.d_head).transpose(1, 2)
        q, k, v = split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if key_padding_mask is not None:
            scores = scores.masked_fill(key_padding_mask.view(B, 1, 1, L), float("-inf"))
        attention = self.dropout(torch.softmax(scores, dim=-1))
        ctx = (attention @ v).transpose(1, 2).contiguous().view(B, L, d)
        return self.out_proj(ctx)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, nhead, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask):
        x = x + self.dropout(self.attn(self.norm1(x), key_padding_mask))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x


class TransformerEncoderModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4,
                 dim_feedforward=512, max_len=512, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model, max_len)
        self.in_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids) * math.sqrt(self.d_model)
        x = self.in_dropout(self.pos(x))
        key_padding_mask = attention_mask == 0
        for block in self.blocks:
            x = block(x, key_padding_mask)
        x = self.final_norm(x)
        return masked_mean(x, attention_mask)


class BagOfEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model=256, pad_id=0, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        return masked_mean(self.dropout(self.embed(input_ids)), attention_mask)

Writing model.py


contrastive_learning.py

In [4]:
%%writefile contrastive_learning.py
import torch
import torch.nn.functional as F


def info_nce(query_emb, passage_emb, temperature=0.05):
    q = F.normalize(query_emb, dim=-1)
    p = F.normalize(passage_emb, dim=-1)
    logits = (q @ p.t()) / temperature
    labels = torch.arange(q.size(0), device=q.device)
    loss_q = F.cross_entropy(logits, labels)
    loss_p = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss_q + loss_p)

def triplet_nce(query_emb, pos_emb, neg_emb, temperature: float = 0.05):
    q   = F.normalize(query_emb, dim=-1)
    pos = F.normalize(pos_emb, dim=-1)
    neg = F.normalize(neg_emb, dim=-1)
    all_p  = torch.cat([pos, neg], dim=0) 
    logits = (q @ all_p.t()) / temperature
    labels = torch.arange(q.size(0), device=q.device)
    return F.cross_entropy(logits, labels)

Writing contrastive_learning.py


load_msmarco.py

In [5]:
%%writefile load_msmarco.py
import json, os
from datasets import load_dataset
import config


def main(max_examples: int = config.MAX_EXAMPLES) -> None:
    os.makedirs(config.DATA_DIR, exist_ok=True)

    print(f"Loading ms_marco v2.1 ({'all' if not max_examples else max_examples} examples)...")
    ds = load_dataset("microsoft/ms_marco", "v2.1", split="train")
    if max_examples:
        ds = ds.select(range(min(max_examples, len(ds))))

    text_to_pid, passages, queries_meta = {}, [], []

    for ex in ds:
        pos_ids, neg_ids = [], []
        for text, selected in zip(ex["passages"]["passage_text"],
                                  ex["passages"]["is_selected"]):
            if text not in text_to_pid:
                text_to_pid[text] = len(passages)
                passages.append({"id": len(passages), "text": text})
            (pos_ids if selected else neg_ids).append(text_to_pid[text])

        if pos_ids:
            queries_meta.append({
                "query_id": ex["query_id"],
                "query":    ex["query"],
                "pos_ids":  pos_ids,
                "neg_ids":  neg_ids,
            })

    with open(config.PASSAGES_PATH, "w", encoding="utf-8") as f:
        json.dump(passages, f, ensure_ascii=False)
    with open(config.QUERIES_META_PATH, "w", encoding="utf-8") as f:
        json.dump(queries_meta, f, ensure_ascii=False)

    print(f"passages: {len(passages):,} | queries: {len(queries_meta):,}")


Writing load_msmarco.py


**build_tokenizer.py** - train BPE tokenizer + load/encode helpers, whole point of this file is to handle rare/unseen words and build an vocabulary

In [6]:
%%writefile build_tokenizer.py
import json, os, tempfile
from typing import Callable, List, Tuple
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import config

SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]


def _text_iterator():
    with open(config.QUERIES_META_PATH, encoding="utf-8") as f:
        for item in json.load(f):
            yield item["query"]
    with open(config.PASSAGES_PATH, encoding="utf-8") as f:
        for item in json.load(f):
            yield item["text"]


def train_tokenizer(vocab_size: int = config.VOCAB_SIZE,
                    out_path: str = config.TOKENIZER_PATH) -> Tokenizer:
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS)

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt",
                                     delete=False, encoding="utf-8") as tmp:
        for text in _text_iterator():
            tmp.write(text.replace("\n", " ") + "\n")
        tmp_path = tmp.name

    tokenizer.train([tmp_path], trainer)
    os.unlink(tmp_path)
    tokenizer.save(out_path)
    print(f"saved tokenizer -> {out_path} (vocab={tokenizer.get_vocab_size()})")
    return tokenizer


def load_tokenizer(path: str = config.TOKENIZER_PATH) -> Tuple[Callable[[str], List[int]], int, int]:
    tokenizer = Tokenizer.from_file(path)
    pad_id = tokenizer.token_to_id("[PAD]")
    vocab_size = tokenizer.get_vocab_size()
    tokenize = lambda text: tokenizer.encode(text).ids
    return tokenize, pad_id, vocab_size


def main() -> None:
    train_tokenizer()

Writing build_tokenizer.py


**data.py** - turns list of token-ID sequennces into padded pytorch batches encoder can consume. Its gonna be a bridge between the tokenizer and the model

In [7]:
%%writefile data.py
from typing import Callable, List, Tuple
import torch
from torch import Tensor
from torch.utils.data import Dataset
import config


class TripletDataset(Dataset):
    """(query, positive, hard negative)"""
    def __init__(self, triples, tokenize: Callable, max_length: int = config.MAX_LEN):
        self.triples, self.tokenize, self.max_length = triples, tokenize, max_length

    def __len__(self): return len(self.triples)

    def __getitem__(self, i):
        q, pos, neg = self.triples[i]
        t, m = self.tokenize, self.max_length
        return t(q)[:m], t(pos)[:m], t(neg)[:m]


class TextDataset(Dataset):
    def __init__(self, texts, tokenize: Callable, max_length: int = config.MAX_LEN):
        self.texts, self.tokenize, self.max_length = texts, tokenize, max_length

    def __len__(self): return len(self.texts)

    def __getitem__(self, i) -> List[int]:
        return self.tokenize(self.texts[i])[:self.max_length]


def pad_batch(sequences: List[List[int]], pad_id: int = 0) -> Tuple[Tensor, Tensor]:
    sequences = [seq if seq else [pad_id] for seq in sequences]
    max_len = max(len(s) for s in sequences)
    ids  = torch.full((len(sequences), max_len), pad_id, dtype=torch.long)
    mask = torch.zeros((len(sequences), max_len), dtype=torch.long)
    for i, seq in enumerate(sequences):
        ids[i, :len(seq)]  = torch.tensor(seq, dtype=torch.long)
        mask[i, :len(seq)] = 1
    return ids, mask


def triplet_collate(batch, pad_id: int = 0):
    queries, pos, neg = zip(*batch)
    return pad_batch(queries, pad_id), pad_batch(pos, pad_id), pad_batch(neg, pad_id)


def text_collate(batch, pad_id: int = 0):
    return pad_batch(batch, pad_id)

Writing data.py


**build_pairs.py** - Turn the book's chunks into training pairs + eval sets. This file is the bridge between raw chunks and what the model actually trains and is scored on

In [8]:
%%writefile build_pairs.py
import json, random
from typing import Dict, List
import config


def main() -> None:
    random.seed(config.SEED)

    with open(config.PASSAGES_PATH, encoding="utf-8") as f:
        passages = json.load(f)
    pid_to_text = {p["id"]: p["text"] for p in passages}
    all_pids = list(pid_to_text.keys())

    with open(config.QUERIES_META_PATH, encoding="utf-8") as f:
        queries_meta = json.load(f)

    indices = list(range(len(queries_meta)))
    random.shuffle(indices)
    n_train = int(0.8 * len(indices))
    n_val   = int(0.1 * len(indices))
    train_idx = indices[:n_train]
    val_idx   = indices[n_train:n_train + n_val]
    test_idx  = indices[n_train + n_val:]

    train_pairs = []
    for idx in train_idx:
        meta = queries_meta[idx]
        pos_id = meta["pos_ids"][0]
        neg_id = (random.choice(meta["neg_ids"]) if meta["neg_ids"]
                  else random.choice(all_pids))
        train_pairs.append([meta["query"], pid_to_text[pos_id], pid_to_text[neg_id]])

    def build_eval(split: List[int]) -> Dict:
        qs, gold = [], []
        for idx in split:
            meta = queries_meta[idx]
            qs.append(meta["query"])
            gold.append(meta["pos_ids"][0])
        return {"queries": qs, "gold": gold}

    with open(config.TRAIN_PAIRS_PATH, "w", encoding="utf-8") as f:
        json.dump(train_pairs, f, ensure_ascii=False)
    with open(config.VAL_EVAL_PATH, "w", encoding="utf-8") as f:
        json.dump(build_eval(val_idx), f, ensure_ascii=False)
    with open(config.TEST_EVAL_PATH, "w", encoding="utf-8") as f:
        json.dump(build_eval(test_idx), f, ensure_ascii=False)

    print(f"train triples: {len(train_pairs):,} | val: {len(val_idx):,} | test: {len(test_idx):,}")

Writing build_pairs.py


In [9]:
import os; os.makedirs("eval", exist_ok=True)

In [10]:
%%writefile eval/metrics.py
from typing import List


def recall_at_k(retrieved_ids: List[int], relevant_id: int, k: int) -> float:
    return 1.0 if relevant_id in retrieved_ids[:k] else 0.0


def reciprocal_rank(retrieved_ids: List[int], relevant_id: int) -> float:
    for i, rid in enumerate(retrieved_ids):
        if rid == relevant_id:
            return 1.0 / (i + 1)
    return 0.0


def mean_reciprocal_rank(retrieved_list: List[List[int]], gold_ids: List[int]) -> float:
    scores = [reciprocal_rank(ret, gold) for ret, gold in zip(retrieved_list, gold_ids)]
    return sum(scores) / len(scores) if scores else 0.0


def macro_recall_at_k(retrieved_list: List[List[int]], gold_ids: List[int], k: int) -> float:
    scores = [recall_at_k(ret, gold, k) for ret, gold in zip(retrieved_list, gold_ids)]
    return sum(scores) / len(scores) if scores else 0.0


Writing eval/metrics.py


**prepare_data.py** - This file runs whole data pipeline in order.

In [11]:
import load_msmarco
import build_tokenizer
import build_pairs

print("1/3 load_msmarco ...")
load_msmarco.main()
print("2/3 training tokenizer ...")
build_tokenizer.main()
print("3/3 building pairs + eval sets ...")
build_pairs.main()
print("done.")

1/3 load_msmarco ...
Loading ms_marco v2.1 (200000 examples)...


README.md: 0.00B [00:00, ?B/s]

v2.1/validation-00000-of-00001.parquet:   0%|          | 0.00/210M [00:00<?, ?B/s]

v2.1/train-00000-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

v2.1/train-00001-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

...

v2.1/test-00000-of-00001.parquet:   0%|          | 0.00/204M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/101093 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/808731 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/101092 [00:00<?, ? examples/s]

passages: 1,898,742 | queries: 122,356
2/3 training tokenizer ...



saved tokenizer -> tokenizer.json (vocab=8000)
3/3 building pairs + eval sets ...
train triples: 97,884 | val: 12,235 | test: 12,237
done.
